# Lecture 10 - Optimization och fine-tuning
**Assignment: Transfer learning and data pipeline tuning**  

Instructions:

1. Use a pretrained model (e.g., ResNet18)
2. Compare frozen vs fine-tuned performance
3. Measure small performance tweaks

## Task 1: Transfer learning
Start with a pretrained model and a new classifier head.

In [1]:
# TODO: Load a pretrained model
import torch
from torchvision import models 

# We download ResNet using torchvision and load the weights
# from the version that is trained on ImageNet

model = models.resnet18(weights = models.ResNet18_Weights.IMAGENET1K_V1)

In [2]:
# We take a look at the model
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [3]:
# TODO: Freeze the base layers
from torch.nn import Linear
num_feats = model.fc.in_features
model.fc = Linear(num_feats, 10)

# Here we set the gradient to false, meaning
# the weights will not be updated during backprop
# We start by freezing all layers
for param in model.parameters():
    param.requires_grad = False

# Then we unfreeze our fully connected layer at the endfor param in model.fc.parameters():
for param in model.fc.parameters():
    param.requires_grad = True

In [4]:
# Move the model to a gpu device

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model.to(device)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
# We need a dataset to train the model on!
# We'll use CIFAR-10

from torchvision import datasets
# The downloaded data is in the form of PIL images, we need to transform them to tensors
from torchvision import transforms
transform = transforms.Compose([
    transforms.ToTensor()
])


train_data = datasets.CIFAR10(root = "data", train = True, download = True, transform = transform)
test_data = datasets.CIFAR10(root = "data", train = False, download = True, transform = transform)


# We also need a data loader
from torch.utils.data import DataLoader

# Shuffle = True for training, False for testing to preserve the order of the test data
train_dataloader = DataLoader(train_data, batch_size = 32, shuffle = True)
test_dataloader = DataLoader(test_data, batch_size = 32, shuffle = False)   

c:\Users\Lilit\Documents\GitHub\ML-Ramverk\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [6]:
# TODO: Train a new classifier head
import torch.optim as optim
import torch.nn as nn

# Instead of passing in model.parameters(),
# we now only pass in model.fc.parameters()
# That is, the parameters in our Fully Connected layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

#Training loop
num_epochs = 1
for _ in range(num_epochs):
    model.train()
    for xb, yb in train_dataloader:
        xb, yb = xb.to(device), yb.to(device)
        # We make sure to move our batch data to the device!        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

In [7]:
# TODO: Record accuracy

def eval_acc(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = torch.argmax(model(xb), dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(total, 1)

train_acc = eval_acc(model, train_dataloader)
test_acc = eval_acc(model, test_dataloader)

print(f"Train Accuracy: {train_acc:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")

Train Accuracy: 0.43
Test Accuracy: 0.42


## Task 2: Fine-tuning
Unfreeze part of the base and compare performance.

In [9]:
# TODO: Unfreeze part of the base

# Now, we unfreeze a part of the base, e.g., layer4
for name, param in model.named_parameters():
    if name.startswith("layer4"):
        param.requires_grad = True

# In the expression below, we use some Python hacks to set all layers
# where parameters.requires_grad = True, and give them a super low learning rate
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0005)

In [10]:
# TODO: Train again and record accuracy

num_epochs = 1
for _ in range(num_epochs):
    model.train()
    for xb, yb in train_dataloader:
        # We make sure to move our batch data to the device!
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
acc_tuned = eval_acc(model, test_dataloader)
print(f"Fine-tuned accuracy: {acc_tuned:.4f}")

Fine-tuned accuracy: 0.6700


In [11]:
# TODO: Compare with Task 1

# TODO: Record accuracy

def eval_acc(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = torch.argmax(model(xb), dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / max(total, 1)

train_acc = eval_acc(model, train_dataloader)
test_acc = eval_acc(model, test_dataloader)

print(f"Train Accuracy: {train_acc:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")

Train Accuracy: 0.73
Test Accuracy: 0.67


In [12]:
# Save model with torch.save and export to ONNX
model.eval()

# 1) Save with torch.save (state_dict for flexible loading)
torch.save(model.state_dict(), "L11_model.pt")
print("Saved PyTorch state_dict to L11_model.pt")

# 2) Export to ONNX (input shape: CIFAR-10 is 3x32x32)
# When we save our model with ONNX, it will
# Try to precompute everything it can
# It usually does this by performing 1 run of the model
# With some form of input data
# Often we define 1 record of random data (exactly the size of the training data)
# But we could send in a concrete example (like test data, 1 row)
dummy_input = torch.randn(1, 3, 32, 32, device=device)

# Here we save the model as an ONNX file
# We only need to pass in a model and an example
# But there are plenty of extra things we can save
torch.onnx.export(
    model,
    dummy_input,
    "L11_model.onnx",
    export_params=True,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
)
print("Exported model to L11_model.onnx")

Saved PyTorch state_dict to L11_model.pt


C:\Users\Lilit\AppData\Local\Temp\ipykernel_7620\423463559.py:20: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0305 10:38:53.716000 7620 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0305 10:38:53.717000 7620 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0305 10:38:53.718000 7620 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'i

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


C:\Users\Lilit\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 41 of general pattern rewrite rules.
Exported model to L11_model.onnx


## Task 3: Dataloader tuning
Measure the effect of data loader settings.

In [ ]:
# FUN FACTS!

#  TODO: Test at least two of:
# - num_workers (good on Linux, okay on Windows, poor on Mac)
# - pin_memory (good on Linux and Windows, poor on Mac)
# - batch size (make as large as possible, start at 32)

# Different computers have different hardware and software
# We generally want to distribute computing power as optimally as possible
# For our ML model. 

# num_workers is a way to divide how many units perform calculations simultaneously
# BUT: on Windows and Mac, the instantiation is of the Spawn type (it creates an entirely new process, takes a long time)
# On Linux, it is of the Fork type, a direct copy, more or less instant
# It's worth keeping track of, at least. 

# Batch size is how much data is fed into the model at once. 
# Batch size is also something that can optimize time efficiency and memory management during training
# We typically want as large a batch as possible
# BUT: too large a batch size can cause the computer to run out of memory
# So we typically start with 32, and then step up in powers of 2 (32, 64, 128, 256, 512, ...)

# Different types of computers have different memory management. 
# On Mac, it's Unified RAM, 
# On Windows, Fixed VRAM
# On Linux, Fixed